# Rebar YOLO26 Segmentation Visualization

This notebook visualizes the trained `yolo26l_sam3_rebar_v1` run, recomputes validation metrics from `weights/best.pt`, and overlays predicted masks on the validation images.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd()
if ROOT.name == "rebar-segementation-yolo26":
    ROOT = ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from ultralytics import YOLO

RUN_DIR = ROOT / "rebar-segementation-yolo26" / "yolo26l_sam3_rebar_v1"
MODEL_PATH = RUN_DIR / "weights" / "best.pt"
RESULTS_IMAGE = RUN_DIR / "results.png"
RESULTS_CSV = RUN_DIR / "results.csv"
DATA_YAML = ROOT / "datasets" / "sam3_annotation_without_open_source_rebar_v1" / "data.yaml"
VALID_IMAGES = ROOT / "datasets" / "sam3_annotation_without_open_source_rebar_v1" / "valid" / "images"

for path in [RUN_DIR, MODEL_PATH, RESULTS_IMAGE, RESULTS_CSV, DATA_YAML, VALID_IMAGES]:
    if not path.exists():
        raise FileNotFoundError(path)

image_paths = sorted(p for p in VALID_IMAGES.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(f"Found {len(image_paths)} validation images")
print(f"Model: {MODEL_PATH}")
print(f"Data:  {DATA_YAML}")

## Training Curves

In [ ]:
display(Image.open(RESULTS_IMAGE))

In [ ]:
results_df = pd.read_csv(RESULTS_CSV)
results_df.columns = results_df.columns.str.strip()

metric_columns = [
    "epoch",
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
    "metrics/precision(M)",
    "metrics/recall(M)",
    "metrics/mAP50(M)",
    "metrics/mAP50-95(M)",
]

display(results_df[metric_columns].tail(1).T.rename(columns={results_df.index[-1]: "final_epoch"}))

## Validation Metrics From `best.pt`

In [ ]:
model = YOLO(str(MODEL_PATH))
metrics = model.val(data=str(DATA_YAML))

In [ ]:
def metrics_to_row(prefix, values):
    return {
        "metric_type": prefix,
        "mAP50-95": values.map,
        "mAP50": values.map50,
        "mAP75": values.map75,
        "per_class_mAP50-95": list(values.maps),
    }

summary = pd.DataFrame([
    metrics_to_row("box", metrics.box),
    metrics_to_row("mask", metrics.seg),
])
display(summary)

for metric_type, values in [("box", metrics.box), ("mask", metrics.seg)]:
    image_metrics = getattr(values, "image_metrics", None)
    if image_metrics is None:
        print(f"{metric_type}.image_metrics is not available in this Ultralytics result object")
    else:
        print(f"{metric_type}.image_metrics")
        display(pd.DataFrame(image_metrics).T if isinstance(image_metrics, dict) else image_metrics)

## Predicted Mask Overlays

In [ ]:
def overlay_masks(image, masks):
    image = image.convert("RGBA")
    if masks is None or len(masks) == 0:
        return image

    masks = 255 * masks.cpu().numpy().astype(np.uint8)
    n_masks = masks.shape[0]
    cmap = matplotlib.colormaps.get_cmap("rainbow").resampled(n_masks)
    colors = [tuple(int(c * 255) for c in cmap(i)[:3]) for i in range(n_masks)]

    for mask, color in zip(masks, colors):
        mask = Image.fromarray(mask)
        if mask.size != image.size:
            mask = mask.resize(image.size, Image.Resampling.NEAREST)
        overlay = Image.new("RGBA", image.size, color + (0,))
        alpha = mask.point(lambda v: int(v * 0.5))
        overlay.putalpha(alpha)
        image = Image.alpha_composite(image, overlay)
    return image

predictions = model.predict(source=[str(p) for p in image_paths], conf=0.25, save=False, verbose=False)
print(f"Predicted {len(predictions)} images")

## Raw Predicted Masks

In [ ]:
# Change this index to inspect masks for a different validation image.
IMAGE_INDEX = 0
MAX_MASKS = 24

result = predictions[IMAGE_INDEX]
image_path = image_paths[IMAGE_INDEX]

if result.masks is None or len(result.masks) == 0:
    print(f"No masks predicted for {image_path.name}")
else:
    masks = result.masks.data.cpu().numpy()
    n_masks = min(len(masks), MAX_MASKS)
    cols = 6
    rows = int(np.ceil(n_masks / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.4, rows * 2.4))
    axes = np.atleast_1d(axes).reshape(rows, cols)

    for ax in axes.ravel():
        ax.axis("off")

    for mask_index, ax in enumerate(axes.ravel()[:n_masks]):
        ax.imshow(masks[mask_index], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"mask {mask_index}", fontsize=9)
        ax.axis("off")

    fig.suptitle(f"{image_path.name}: showing {n_masks} of {len(masks)} masks", fontsize=12)
    plt.tight_layout()
    display(fig)
    plt.close(fig)

In [ ]:
cols = 3
rows = int(np.ceil(len(predictions) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4.5))
axes = np.atleast_1d(axes).reshape(rows, cols)

for ax in axes.ravel():
    ax.axis("off")

for ax, result, image_path in zip(axes.ravel(), predictions, image_paths):
    image = Image.open(image_path)
    masks = result.masks.data if result.masks is not None else None
    overlay = overlay_masks(image, masks)
    mask_count = 0 if masks is None else len(masks)
    ax.imshow(overlay)
    ax.set_title(f"{image_path.name}\n{mask_count} masks", fontsize=9)
    ax.axis("off")

plt.tight_layout()
display(fig)
plt.close(fig)